# 2026 — Pre-Training Without Learning-Rate Decay: Constant LR + Checkpoint Averaging

**Sources:** Sebastian Raschka, *"Pre-Training Without Learning Rate Decay"* (November 17, 2026), surveying recent recipes. **Kolibri-1** (78B, October 2026) and DatologyAI's models train with a **constant** learning rate and **average the last checkpoints** instead of decaying. Background: Warmup-Stable-Decay (WSD; MiniCPM 2024), "Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations" (Hägele et al. 2024), and weight averaging (Polyak; LAWA; model soups).

Schedules used by 2026 open models (from Raschka's note):

| schedule | how it works | used by |
|---|---|---|
| **cosine** | warm up, then decay smoothly to ~0 by a horizon fixed in advance | Llama 3, OLMo 2/3, GLM-4.5/5, Kimi K3, Phi-4 |
| **WSD** | warm up, hold constant, decay quickly over the last ~10–20% | DeepSeek V3/V4, Kimi K2 / Kimi Linear, Nemotron 3, SmolLM3, MiniMax M2 |
| **constant + averaging** | never decay; report the *average of recent checkpoints* | Kolibri-1, DatologyAI |

**Why it works:** with a constant LR, SGD-type optimizers bounce around a valley floor with noise proportional to the LR. Decaying the LR shrinks the bouncing. **Averaging the iterates** cancels the bouncing directly, so you get most of the decay's benefit without lowering the LR and without fixing the run length in advance. Training can continue from the constant-LR weights at any time, so every checkpoint is "anytime".

We train the same small GPT with each schedule and track validation loss throughout.

In [ ]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)

In [ ]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")

In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj, self.h = nn.Linear(d, 3 * d), nn.Linear(d, d), heads
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    """A small GPT-2-style character model (pre-norm blocks, learned positions)."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return self.head(self.ln(x))

def train_gpt(model, train_data, steps=800, B=32, lr=2e-3):
    model = model.to(train_data.device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - model.ctx - 1, (B,), device=train_data.device)
        x = torch.stack([train_data[i:i + model.ctx] for i in ix]); y = torch.stack([train_data[i + 1:i + model.ctx + 1] for i in ix])
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def eval_loss(model, data, n=20, B=32):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    for _ in range(n):
        ix = torch.randint(0, len(data) - model.ctx - 1, (B,), generator=g).to(data.device)
        x = torch.stack([data[i:i + model.ctx] for i in ix]); y = torch.stack([data[i + 1:i + model.ctx + 1] for i in ix])
        losses.append(F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1)).item())
    return float(np.mean(losses))

In [ ]:
STEPS, PEAK, WARM = 2000, 3e-3, 100
def lr_at(s, kind):
    if s < WARM: return PEAK * (s + 1) / WARM
    if kind == "cosine": return PEAK * 0.5 * (1 + math.cos(math.pi * (s - WARM) / (STEPS - WARM)))
    if kind == "wsd":
        d0 = int(0.8 * STEPS)
        return PEAK if s < d0 else PEAK * (1 - (s - d0) / (STEPS - d0))
    return PEAK                                                                     # constant

@torch.no_grad()
def val_loss(model, n=10):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    model.eval()
    for _ in range(n):
        ix = torch.randint(0, len(val_data) - 129, (32,), generator=g).to(device)
        x = torch.stack([val_data[i:i + 128] for i in ix]); y = torch.stack([val_data[i + 1:i + 129] for i in ix])
        losses.append(F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item())
    model.train(); return float(np.mean(losses))

def run(kind, eval_every=100, avg_every=10, avg_window=400, ema_decay=0.995):
    seed_everything(0); model = GPT(V, d=128, n_layers=4).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=PEAK, weight_decay=0.1, betas=(0.9, 0.95))
    ema = [p.detach().clone() for p in model.parameters()]
    snaps = []                                                                        # checkpoints for the windowed average
    curves = {"raw": [], "ema": [], "window": []}; t0 = time.time()
    for s in range(STEPS):
        for g_ in opt.param_groups: g_["lr"] = lr_at(s, kind)
        ix = torch.randint(0, len(train_data) - 129, (32,), device=device)
        x = torch.stack([train_data[i:i + 128] for i in ix]); y = torch.stack([train_data[i + 1:i + 129] for i in ix])
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        with torch.no_grad():
            for e, p in zip(ema, model.parameters()): e.lerp_(p, 1 - ema_decay)
        if (s + 1) % avg_every == 0:
            snaps.append([p.detach().clone() for p in model.parameters()])
            snaps = snaps[-(avg_window // avg_every):]
        if (s + 1) % eval_every == 0:
            curves["raw"].append(val_loss(model))
            backup = [p.detach().clone() for p in model.parameters()]
            for name, weights in [("ema", ema), ("window", [torch.stack(ws).mean(0) for ws in zip(*snaps)])]:
                with torch.no_grad():
                    for p, w in zip(model.parameters(), weights): p.copy_(w)
                curves[name].append(val_loss(model))
            with torch.no_grad():
                for p, w in zip(model.parameters(), backup): p.copy_(w)
    return curves, time.time() - t0

results = {}
for kind in ("cosine", "wsd", "constant"):
    results[kind], sec = run(kind)
    c = results[kind]
    print(f"{kind:8s} final val loss: raw weights {c['raw'][-1]:.4f} | EMA {c['ema'][-1]:.4f} | average of last 400 steps {c['window'][-1]:.4f}   ({sec:.0f}s)")

In [ ]:
steps_axis = np.arange(100, STEPS + 1, 100)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot([lr_at(s, "cosine") for s in range(STEPS)], label="cosine")
axes[0].plot([lr_at(s, "wsd") for s in range(STEPS)], label="WSD"); axes[0].plot([lr_at(s, "constant") for s in range(STEPS)], label="constant")
axes[0].set(xlabel="step", ylabel="learning rate", title="Schedules"); axes[0].legend()
axes[1].plot(steps_axis, results["cosine"]["raw"], label="cosine (raw)")
axes[1].plot(steps_axis, results["wsd"]["raw"], label="WSD (raw)")
axes[1].plot(steps_axis, results["constant"]["raw"], label="constant (raw)", alpha=0.6)
axes[1].plot(steps_axis, results["constant"]["window"], label="constant + checkpoint average", lw=2)
axes[1].plot(steps_axis, results["constant"]["ema"], "--", label="constant + EMA")
axes[1].set(xlabel="step", ylabel="validation loss", title="Validation loss during training", ylim=(None, results["constant"]["raw"][3])); axes[1].legend()
plt.tight_layout(); plt.show()

## The "anytime" property
Suppose you must stop at step 1000 instead of 2000. A cosine schedule tuned for 2000 steps is still at a high LR at step 1000, so you'd have to rerun with a new horizon. Constant LR + averaging can hand over a good model at any point, and WSD only needs a short decay from any stable checkpoint.

In [ ]:
i = 1000 // 100 - 1
print(f"validation loss at step 1000: cosine(2000-step horizon) raw {results['cosine']['raw'][i]:.4f} | constant raw {results['constant']['raw'][i]:.4f} | constant + average {results['constant']['window'][i]:.4f}")

## Takeaways
- With a constant LR, the raw weights are noisy. Averaging recent checkpoints (or an EMA) removes most of that noise, and typically lands close to a decayed schedule's final loss, without having to decide the run length in advance.
- WSD is the middle ground most 2026 frontier labs chose: train at constant LR (so you can branch or extend anytime), then decay briefly at the end.
- The exact ranking between cosine, WSD and averaging at this toy scale depends on the seed and the averaging window. The robust observation is the size of the gap between "constant, raw" and "constant, averaged".
- This is a training-recipe change, not an architectural one. It made Raschka's list because it simplified how 2026 models like Kolibri-1 were trained.